ベース：黒地図（CARTO Dark Matter）

国境線：
HDX OCHA Syrian Arab Republic – Subnational Administrative Boundaries

地域線：
HDX OCHA Syrian Arab Republic – Subnational Administrative Boundaries

人口ラスタ：
WorldPop Open Population Data 2026

解析内容：
Zonal Statistics

対象地域：
・Aleppo
・Idleb
・Lattakia

GeoPandasポリゴン（Governorate）とWorldPop人口ラスタを利用し、

各Governorate内部の推定人口合計を集計。

集計結果をProportional Symbol Mapとして可視化。

使用技術：
・Rasterio
・Rasterstats
・GeoPandas
・Folium

In [1]:
!pip install rasterstats

zsh:1: command not found: pip


In [2]:
import geopandas as gpd

import rasterio

import folium
from rasterstats import zonal_stats

# ETL Extract
# 人口ラスタと行政区ポリゴンを読み込む
admin1 = gpd.read_file("syr_admin1.geojson")

with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:
    print(src.crs)
    print(src.bounds)

/Users/marisa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


EPSG:4326
BoundingBox(left=35.7166658038, bottom=32.310833540089995, right=42.37666577716, top=37.320000186719994)


In [3]:
# ETL Transform
# Zonal Statistics
# PolygonごとにRaster人口を集計

target_admins = admin1[
    admin1["adm1_name"].isin([
        "Aleppo",
        "Idleb",
        "Lattakia"
    ])
]

stats = zonal_stats(
    target_admins,
    "tif_1_syria_worldpop_2026.tif",
    stats=["sum"],
    geojson_out=True
)

print(stats)

[{'id': '1', 'type': 'Feature', 'properties': {'adm1_name': 'Aleppo', 'adm1_name1': 'حلب', 'adm1_name2': None, 'adm1_name3': None, 'adm1_pcode': 'SY02', 'adm0_name': 'Syrian Arab Republic', 'adm0_name1': 'الجمهورية العربية السورية', 'adm0_name2': None, 'adm0_name3': None, 'adm0_pcode': 'SY', 'valid_on': Timestamp('2020-12-17 00:00:00'), 'valid_to': None, 'area_sqkm': 19891.01295691, 'version': 'v02', 'lang': 'en', 'lang1': 'ar', 'lang2': None, 'lang3': None, 'adm1_ref_name': 'Aleppo', 'center_lat': 36.14689671, 'center_lon': 37.39516317, 'sum': 4550203.0}, 'geometry': {'type': 'Polygon', 'coordinates': (((38.2963695520001, 36.90689277700017), (38.295602799000164, 36.90694999700008), (38.290508610000074, 36.90804634800014), (38.28212928800008, 36.909849167000175), (38.27839097400005, 36.91065444499998), (38.26902198800008, 36.91267204300016), (38.26570285900016, 36.91337204800004), (38.26524162300018, 36.91346931400011), (38.26189844300012, 36.91347478500012), (38.2581806180001, 36.9134

• 人口合計 sum
• Governoratez
• geometry
• polygon座標
• metadata
etc. 全部抽出されている状態

In [4]:
# ETL Transform
# Raster解析
# zonal statistics結果から人口合計を抽出

for area in stats:
    print(area["properties"]["adm1_name"])
    print(area["properties"]["sum"])

Aleppo
4550203.0
Idleb
1746525.75
Lattakia
1519531.0


In [5]:
# ETL Transform
# zonal statistics結果から、地図描画用データへ整理

population_results = []

for area in stats:
    props = area["properties"]
    population_results.append({
        "name": props["adm1_name"],
        "population": props["sum"],
        "lat": props["center_lat"],
        "lon": props["center_lon"]
    })

print(population_results)

[{'name': 'Aleppo', 'population': 4550203.0, 'lat': 36.14689671, 'lon': 37.39516317}, {'name': 'Idleb', 'population': 1746525.75, 'lat': 35.85769678, 'lon': 36.5708795}, {'name': 'Lattakia', 'population': 1519531.0, 'lat': 35.58078003, 'lon': 35.99093035}]


In [6]:
# ETL Transform
# 人口値をCircleMarkerの半径へ変換

for item in population_results:
    item["radius"] = item["population"] / 300000

print(population_results)

[{'name': 'Aleppo', 'population': 4550203.0, 'lat': 36.14689671, 'lon': 37.39516317, 'radius': 15.167343333333333}, {'name': 'Idleb', 'population': 1746525.75, 'lat': 35.85769678, 'lon': 36.5708795, 'radius': 5.8217525}, {'name': 'Lattakia', 'population': 1519531.0, 'lat': 35.58078003, 'lon': 35.99093035, 'radius': 5.065103333333333}]


In [7]:
# Map Setup
# 黒地図ベースを作成

tiles_dark_nolabels = 'https://{s}.basemaps.cartocdn.com/dark_nolabels/{z}/{x}/{y}{r}.png'
attr_dark = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'

m = folium.Map(
    location=[35.8, 37.0],
    zoom_start=7,
    tiles=tiles_dark_nolabels,
    attr=attr_dark
)

In [8]:
# ETL Load
# 国境線・地域線を地図へ追加

folium.GeoJson(
    'syr_admin0.geojson',
    name='Country',
    style_function=lambda x: {'color': 'white', 'weight': 1, 'fillOpacity': 0}
).add_to(m)

folium.GeoJson(
    'syr_admin1.geojson',
    name='Governorate',
    style_function=lambda x: {'color': 'white', 'weight': 1, 'fillOpacity': 0}
).add_to(m)

In [9]:
# ETL Transform
# Folium表示用に必要な列だけ残す

target_admins_map = target_admins[["adm1_name", "geometry"]]

In [10]:
# ETL Load
# 対象3州を薄く塗る

folium.GeoJson(
    target_admins_map,
    style_function=lambda x: {
        "fillColor": "#b7d86f",
        "fillOpacity": 0.15,
        "color": "white",
        "weight": 1
    }
).add_to(m)

In [11]:
# シリア14地域の名入れ
# この地図専用で、微妙に位置変え

governorates = {
    "Aleppo": [36.3, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.7, 36.7],
    "Lattakia": [35.7, 36.0],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1]
}

for name, coords in governorates.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="
                font-size: 12pt; 
                color: white; 
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
                ">{name}</div>'''
        )
    ).add_to(m)

In [12]:
# ETL Load
# 人口合計を円の大きさで地図へ可視化する

for item in population_results:
    folium.CircleMarker(
        location=[item["lat"], item["lon"]],
        radius=item["radius"],
        color="#b7d86f",
        fill=True,
        fillColor="#b7d86f",
        fillOpacity=0.65,
        weight=2,
        popup=f"{item['name']}: {item['population']:,.0f} people"
    ).add_to(m)

In [13]:
# ETL Load
# タイトル・説明ボックスをHTMLとして地図に追加

title_html = '''
             <div style="position: fixed; 
                         top: 20px; left: 50px; width: 390px; height: 165px; 
                         background-color: rgba(0,0,0,0.75); color: white;
                         z-index:9000; font-size:14px;
                         border:1px solid white; border-radius:8px; padding: 12px;
                         box-shadow: 0 0 15px rgba(0,0,0,0.5);">
             <b style="font-size: 16px;">Northwest Syria</b><br>
             <span style="color: #b7d86a;">Estimated Population by Governorate (WorldPop 2026)</span><br>
             <small style="display: block; margin-top: 6px; line-height: 1.3; color: #eee;">
                Population totals were calculated from WorldPop 2026 raster data using zonal statistics.
                Circle size represents estimated population by governorate.
             </small>
             <div style="margin-top: 10px; font-size: 11px; color: #bbb; border-top: 1px solid #444; padding-top: 6px;">
             Source: <a href="https://hub.worldpop.org/geodata/summary?id=75632" target="_blank" style="color: #3498db; text-decoration: none;">WorldPop 2026</a><br>
             Method: Zonal Statistics / Proportional Symbol Map
             </div>
             </div>
             '''

m.get_root().html.add_child(folium.Element(title_html))

In [14]:
for item in population_results:
    folium.Marker(
        location=[item["lat"], item["lon"]],
        icon=folium.DivIcon(
            html=f'''
            <div style="
                color:white;
                font-size:12pt;
                font-weight:bold;
                white-space: nowrap;
                text-shadow: 0 0 4px black;
            ">
            {item["population"]/1000000:.2f} M
            </div>
            '''
        )
    ).add_to(m)

In [15]:
legend_html = '''
<div style="
position: fixed;
bottom: 40px;
right: 40px;
width: 180px;
background-color: rgba(0,0,0,0.75);
color: white;
border: 1px solid white;
border-radius: 8px;
padding: 10px;
font-size: 12px;
z-index: 9999;
">

<b>Legend</b><br><br>

● Circle size<br>
= Population<br><br>

Source:<br>
WorldPop 2026

</div>
'''

m.get_root().html.add_child(
    folium.Element(legend_html)
)

In [16]:
m.save("07_syria_dmap_NW3_zonalstatistics.html")